# Jaguar Re-Identification: Step 1 - Dataset Exploration

This notebook covers Step 1 of the project: Understanding the Dataset Structure.

We will:
1. Load and explore the training and test datasets
2. Analyze class distribution and data imbalance
3. Examine image variations (angles, lighting, quality)
4. Visualize sample images for different jaguars
5. Identify challenges in the data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load CSV Data and Explore Structure

In [ ]:
# Define data paths
DATA_DIR = Path('/Users/srikaruv/Desktop/Projects/Jaguar_Re-Identification/jaguar-re-id')

# Load CSV files
train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')
sample_submission_df = pd.read_csv(DATA_DIR / 'sample_submission.csv')

print("="*60)
print("TRAINING SET")
print("="*60)
print(f"Shape: {train_df.shape}")
print(f"\nFirst few rows:")
print(train_df.head())
print(f"\nColumn names: {train_df.columns.tolist()}")
print(f"\nData types:\n{train_df.dtypes}")
print(f"\nMissing values:\n{train_df.isnull().sum()}")

print("\n" + "="*60)
print("TEST SET")
print("="*60)
print(f"Shape: {test_df.shape}")
print(f"\nFirst few rows:")
print(test_df.head())
print(f"\nColumn names: {test_df.columns.tolist()}")

print("\n" + "="*60)
print("SAMPLE SUBMISSION")
print("="*60)
print(f"Shape: {sample_submission_df.shape}")
print(f"First few rows:")
print(sample_submission_df.head())

## 2. Analyze Class Distribution & Data Imbalance

This is critical to understand the long-tail distribution challenge.

In [ ]:
# Identify the ID column name (could be 'ID', 'jaguar_id', 'identity', etc.)
id_column = [col for col in train_df.columns if col.lower() in ['id', 'jaguar_id', 'identity', 'class']][0] if any(col.lower() in ['id', 'jaguar_id', 'identity', 'class'] for col in train_df.columns) else train_df.columns[1]

print(f"ID column identified: {id_column}\n")

# Class distribution
class_distribution = train_df[id_column].value_counts().sort_values(ascending=False)

print("="*60)
print("CLASS DISTRIBUTION IN TRAINING SET")
print("="*60)
print(f"Number of unique jaguars: {len(class_distribution)}")
print(f"\nImages per jaguar:")
print(class_distribution)

print("\n" + "="*60)
print("DISTRIBUTION STATISTICS")
print("="*60)
print(f"Total training images: {len(train_df)}")
print(f"Mean images per jaguar: {class_distribution.mean():.2f}")
print(f"Median images per jaguar: {class_distribution.median():.2f}")
print(f"Min images per jaguar: {class_distribution.min()}")
print(f"Max images per jaguar: {class_distribution.max()}")
print(f"Std deviation: {class_distribution.std():.2f}")
print(f"\nImbalance ratio (max/min): {class_distribution.max() / class_distribution.min():.2f}x")

# Show the most and least represented jaguars
print("\n" + "-"*60)
print("TOP 10 Most Photographed Jaguars:")
print("-"*60)
print(class_distribution.head(10))

print("\n" + "-"*60)
print("TOP 10 Least Photographed Jaguars:")
print("-"*60)
print(class_distribution.tail(10))

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Bar plot of class distribution
ax1 = axes[0, 0]
class_distribution.plot(kind='bar', ax=ax1, color='steelblue')
ax1.set_title('Images per Jaguar (All Classes)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Jaguar ID')
ax1.set_ylabel('Number of Images')
ax1.tick_params(axis='x', rotation=45)

# 2. Histogram of class sizes
ax2 = axes[0, 1]
ax2.hist(class_distribution, bins=20, color='coral', edgecolor='black')
ax2.set_title('Distribution of Class Sizes', fontsize=12, fontweight='bold')
ax2.set_xlabel('Number of Images per Jaguar')
ax2.set_ylabel('Frequency')
ax2.axvline(class_distribution.mean(), color='red', linestyle='--', label=f'Mean: {class_distribution.mean():.2f}')
ax2.axvline(class_distribution.median(), color='green', linestyle='--', label=f'Median: {class_distribution.median():.2f}')
ax2.legend()

# 3. Cumulative distribution
ax3 = axes[1, 0]
sorted_counts = np.sort(class_distribution.values)
cumsum = np.cumsum(sorted_counts)
cumsum_pct = (cumsum / cumsum[-1]) * 100
ax3.plot(range(len(cumsum)), cumsum_pct, marker='o', linewidth=2, markersize=4, color='darkgreen')
ax3.set_title('Cumulative Distribution of Training Data', fontsize=12, fontweight='bold')
ax3.set_xlabel('Number of Jaguars (sorted by images count)')
ax3.set_ylabel('Cumulative % of Training Data')
ax3.grid(True, alpha=0.3)

# 4. Box plot
ax4 = axes[1, 1]
ax4.boxplot([class_distribution.values], vert=True, patch_artist=True)
ax4.set_title('Box Plot of Images per Jaguar', fontsize=12, fontweight='bold')
ax4.set_ylabel('Number of Images')
ax4.set_xticklabels(['Training Data'])

plt.tight_layout()
plt.show()

print("\nKey Insight: The data shows significant class imbalance,")
print("which is a major challenge for the model training phase.")

## 3. Explore Image Files and Directory Structure

In [ ]:
# Explore directory structure
train_img_dir = DATA_DIR / 'train' / 'train'
test_img_dir = DATA_DIR / 'test' / 'test'

print("="*60)
print("DIRECTORY STRUCTURE")
print("="*60)
print(f"Training images directory: {train_img_dir}")
print(f"Test images directory: {test_img_dir}")

# Count files in training directory
train_files = list(train_img_dir.glob('*'))
test_files = list(test_img_dir.glob('*'))

print(f"\nTraining images found: {len(train_files)}")
print(f"Test images found: {len(test_files)}")

# Check if image count matches CSV
print(f"Training CSV rows: {len(train_df)}")
print(f"Test CSV rows: {len(test_df)}")

# Get image filename column
img_column = [col for col in train_df.columns if col.lower() in ['image', 'image_id', 'filename', 'image_path']][0] if any(col.lower() in ['image', 'image_id', 'filename', 'image_path'] for col in train_df.columns) else train_df.columns[0]
print(f"\nImage filename column: {img_column}")

# Sample a few filenames
print(f"\nSample training images (first 10):")
print(train_df[img_column].head(10).values)

In [ ]:
# Analyze image properties
print("="*60)
print("IMAGE PROPERTIES ANALYSIS")
print("="*60)

image_properties = []

# Sample images from training set
for idx, row in train_df.head(100).iterrows():
    img_path = train_img_dir / row[img_column]
    if img_path.exists():
        try:
            img = Image.open(img_path)
            image_properties.append({
                'filename': row[img_column],
                'jaguar_id': row[id_column],
                'width': img.width,
                'height': img.height,
                'size_mb': img_path.stat().st_size / (1024 * 1024),
                'mode': img.mode
            })
        except Exception as e:
            print(f"Error loading {img_path}: {e}")

properties_df = pd.DataFrame(image_properties)

if len(properties_df) > 0:
    print(f"\nSampled {len(properties_df)} images for analysis")
    print(f"\nImage dimensions:")
    print(f"  Width: {properties_df['width'].min()}-{properties_df['width'].max()} px (Mean: {properties_df['width'].mean():.0f})")
    print(f"  Height: {properties_df['height'].min()}-{properties_df['height'].max()} px (Mean: {properties_df['height'].mean():.0f})")
    print(f"\nImage file sizes:")
    print(f"  Min: {properties_df['size_mb'].min():.2f} MB")
    print(f"  Max: {properties_df['size_mb'].max():.2f} MB")
    print(f"  Mean: {properties_df['size_mb'].mean():.2f} MB")
    print(f"\nImage modes (color spaces): {properties_df['mode'].unique()}")
    
    # Visualize distribution
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    axes[0].scatter(properties_df['width'], properties_df['height'], alpha=0.6, s=50)
    axes[0].set_title('Image Dimensions Distribution', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Width (pixels)')
    axes[0].set_ylabel('Height (pixels)')
    axes[0].grid(True, alpha=0.3)
    
    axes[1].hist(properties_df['size_mb'], bins=20, color='skyblue', edgecolor='black')
    axes[1].set_title('Image File Size Distribution', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('File Size (MB)')
    axes[1].set_ylabel('Frequency')
    
    plt.tight_layout()
    plt.show()
else:
    print("Could not load images for analysis.")

## 4. Visualize Sample Images

Let's look at some actual jaguar images to understand the variations.

In [ ]:
# Visualize sample images from different jaguars
# Select jaguars with most and least images
most_photographed = class_distribution.index[0]
least_photographed = class_distribution.index[-1]
mid_photographed = class_distribution.index[len(class_distribution)//2]

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
fig.suptitle('Sample Images: Diversity in Jaguar Photography', fontsize=14, fontweight='bold')

# Most photographed jaguar
most_photos = train_df[train_df[id_column] == most_photographed][img_column].head(3)
for idx, img_file in enumerate(most_photos):
    img_path = train_img_dir / img_file
    if img_path.exists():
        img = Image.open(img_path)
        axes[0, idx].imshow(img)
        axes[0, idx].set_title(f'Most Photographed ({most_photographed})\n{img_file}', fontsize=9)
        axes[0, idx].axis('off')

# Mid-level jaguar
mid_photos = train_df[train_df[id_column] == mid_photographed][img_column].head(3)
for idx, img_file in enumerate(mid_photos):
    img_path = train_img_dir / img_file
    if img_path.exists():
        img = Image.open(img_path)
        axes[1, idx].imshow(img)
        axes[1, idx].set_title(f'Mid-Frequency ({mid_photographed})\n{img_file}', fontsize=9)
        axes[1, idx].axis('off')

# Least photographed jaguar
least_photos = train_df[train_df[id_column] == least_photographed][img_column].head(3)
for idx, img_file in enumerate(least_photos):
    img_path = train_img_dir / img_file
    if img_path.exists():
        img = Image.open(img_path)
        axes[2, idx].imshow(img)
        axes[2, idx].set_title(f'Least Photographed ({least_photographed})\n{img_file}', fontsize=9)
        axes[2, idx].axis('off')

plt.tight_layout()
plt.show()

print(f"Most photographed: {most_photographed} ({class_distribution[most_photographed]} images)")
print(f"Mid-level: {mid_photographed} ({class_distribution[mid_photographed]} images)")
print(f"Least photographed: {least_photographed} ({class_distribution[least_photographed]} images)")

## 5. Key Findings Summary - Step 1 Analysis

### Dataset Overview
- **Training samples**: [See output above]
- **Test samples**: [See output above]
- **Number of unique jaguar identities**: [See output above]

### Key Challenges Identified

1. **Severe Class Imbalance**
   - Some jaguars have dozens of photos while others have very few
   - This makes standard metrics biased toward well-photographed individuals
   - Solution: Use Identity-Balanced Mean Average Precision (mAP)

2. **Image Variability**
   - Different lighting conditions (daylight, dusk, shadows)
   - Various camera angles and distances
   - Different viewing angles (frontal, lateral, rear)
   - Partial occlusion by vegetation
   - Variable image quality and resolution

3. **Intra-class Variation**
   - Same jaguar looks different under different conditions
   - Makes learning robust features essential

4. **Inter-class Similarity**
   - Different jaguars may have similar spot patterns
   - Requires fine-grained feature discrimination

### Next Steps (Step 2)
1. **Data Augmentation**: Implement geometric and photometric augmentations
2. **Data Normalization**: Standardize image preprocessing
3. **Train/Validation Split**: Create balanced splits considering the imbalance
4. **Feature Extraction Setup**: Prepare for vision model backbone selection